# File Handling 
---

## Table of Contents

1. [Part 1: File Handling](#part-1-file-handling)
   - [1.1 File Handling kya hota hai](#11-file-handling-kya-hota-hai)
   - [1.2 Text vs Binary Files](#12-text-vs-binary-files)
   - [1.3 File Opening aur Modes](#13-file-opening-aur-modes)
   - [1.4 `with` Statement — Best Practice](#14-with-statement--best-practice)
   - [1.5 Files Padhna (Reading)](#15-files-padhna-reading)
   - [1.6 Files Likhna (Writing)](#16-files-likhna-writing)
   - [1.7 File Pointer — `tell()` aur `seek()`](#17-file-pointer--tell-aur-seek)
   - [1.8 Exception Handling in File Ops](#18-exception-handling-in-file-ops)
   - [1.9 Paths ke saath kaam karna](#19-paths-ke-saath-kaam-karna)
   - [1.10 CSV Files Handle Karna](#110-csv-files-handle-karna)
   - [1.11 Advanced: Custom Context Managers](#111-advanced-custom-context-managers)
   - [1.12 Advanced: Bade Files aur Buffering](#112-advanced-bade-files-aur-buffering)
   - [1.13 Binary Files (Images etc.)](#113-binary-files-images-etc)
2. [Part 2: Serialization & Deserialization](#part-2-serialization--deserialization)
   - [2.1 Serialization kya hai](#21-serialization-kya-hai)
   - [2.2 Deserialization kya hai](#22-deserialization-kya-hai)
   - [2.3 Kyun zaroori hai](#23-kyun-zaroori-hai)
   - [2.4 JSON Serialization Deep Dive](#24-json-serialization-deep-dive)
   - [2.5 Custom Objects ko JSON mein Convert Karna](#25-custom-objects-ko-json-mein-convert-karna)
   - [2.6 Pickle Serialization Deep Dive](#26-pickle-serialization-deep-dive)
   - [2.7 JSON vs Pickle — Comparison](#27-json-vs-pickle--comparison)
   - [2.8 Security Warning ⚠️](#28-security-warning)
   - [2.9 Dusre Formats — Brief Overview](#29-dusre-formats--brief-overview)
   - [2.10 Real World Use Case](#210-real-world-use-case)
   - [2.11 Best Practices Cheat Sheet](#211-best-practices-cheat-sheet)

---

# Part 1: File Handling

## 1.1 File Handling kya hota hai

Jab bhi tum koi program band karte ho, RAM mein stored saara data **khatam ho jaata hai** — variables, lists, objects, sab gayab. Agar tumhe data **permanently save** karna hai (disk pe), to uske liye **files** ka use hota hai.

**File Handling** matlab: file ko **open** karna, usme data **read/write** karna, aur phir usko **close** karna — taaki data disk pe surakshit reh jaaye.

**Real life analogy:** Socho tumhare paas ek **diary** hai.
- Diary **kholna** = file open karna
- Diary mein **likhna ya padhna** = read/write karna
- Diary **band karna** = file close karna (taaki likha hua safe rahe)

```mermaid
flowchart LR

    A["🧠 RAM - Temporary Memory"]
    -->|Program band hote hi| B["❌ Data khatam"]

    C["💾 Disk / File - Permanent Storage"]
    -->|Program band hone ke baad bhi| D["✅ Data safe rehta hai"]

    %% Styling
    classDef temporary fill:#eaf2ff,stroke:#3b82f6,color:#111827,stroke-width:2px
    classDef permanent fill:#f0e9ff,stroke:#7c3aed,color:#111827,stroke-width:2px
    classDef danger fill:#fff0f0,stroke:#ef4444,color:#111827,stroke-width:2px
    classDef success fill:#eaf8ef,stroke:#22c55e,color:#111827,stroke-width:2px

    class A temporary
    class C permanent
    class B danger
    class D success

    %% Link styling
    linkStyle default stroke:#64748b,stroke-width:2px
```

---

## 1.2 Text vs Binary Files

Files do tarah ki hoti hain:

| Type | Kya hota hai | Example | Kaise open hoti hai |
|---|---|---|---|
| **Text File** | Human-readable characters (ASCII/Unicode) | `.txt`, `.csv`, `.json`, `.py` | Mode mein `t` (default) |
| **Binary File** | Raw bytes (0s aur 1s), directly readable nahi | `.jpg`, `.mp4`, `.exe`, `.pkl` | Mode mein `b` |

```mermaid
flowchart TD
    A[File] --> B[Text File]
    A --> C[Binary File]
    B --> B1["Notepad mein khol sakte ho\nExample: notes.txt"]
    C --> C1["Special program chahiye\nExample: photo.jpg"]
```

> **Yaad rakho:** Text file bhi internally bytes hi hoti hai, bas encoding (jaise UTF-8) use karke usko characters mein convert kiya jaata hai jab tum padhte ho.

---

## 1.3 File Opening aur Modes

Python mein file open karne ke liye `open()` function use hota hai:

```python
file = open("notes.txt", "r")   # filename, mode
```

### Sabhi Modes ki Table

| Mode | Naam | Kya karta hai | File na ho to | File exist kare to |
|---|---|---|---|---|
| `r` | Read | Sirf padhne ke liye | Error deta hai | Content start se padhta hai |
| `w` | Write | Sirf likhne ke liye | Nayi file banata hai | **Purana content delete** karke naya likhta hai |
| `a` | Append | End mein add karne ke liye | Nayi file banata hai | Content end mein add hota hai |
| `x` | Exclusive create | Nayi file banane ke liye | Nayi file banata hai | **Error** deta hai (already exists) |
| `r+` | Read + Write | Dono kar sakte ho | Error deta hai | Start se read/write |
| `w+` | Write + Read | Dono, par pehle clear | Nayi file banata hai | Purana content delete |
| `a+` | Append + Read | Dono, likhna end mein | Nayi file banata hai | Read start se, write end mein |
| `rb`, `wb`, `ab` | Binary versions | Upar wale hi, par binary mode mein | — | — |

```mermaid
flowchart TD
    Start([Mode choose karna hai?]) --> Q1{File padhni hai ya likhni?}
    Q1 -->|Sirf padhni hai| R["'r' mode"]
    Q1 -->|Likhni hai| Q2{Purana data rakhna hai?}
    Q2 -->|Nahi, overwrite karo| W["'w' mode"]
    Q2 -->|Haan, end mein add karo| A["'a' mode"]
    Q1 -->|Dono karne hain| Q3{File already exist karti hai?}
    Q3 -->|Haan| RPlus["'r+' mode"]
    Q3 -->|Nahi ya matter nahi| WPlus["'w+' mode"]
```

---

## 1.4 `with` Statement — Best Practice

Agar tum manually `open()` use karte ho, to file **close() karna mat bhoolo**, warna:
- Data disk pe properly save nahi hota (buffer mein atka reh sakta hai)
- File "locked" reh sakti hai
- Memory leak ho sakta hai

```python
# ❌ Purana / risky tareeka
file = open("notes.txt", "r")
content = file.read()
file.close()   # agar error aaya beech mein, to ye line kabhi chalegi hi nahi!
```

Isliye **hamesha `with` statement use karo** — ye **context manager** hai jo automatically file close kar deta hai, chahe error aaye ya na aaye.

```python
# ✅ Best practice
with open("notes.txt", "r") as file:
    content = file.read()
    print(content)
# yahan tak aate-aate file automatically close ho chuki hai
```

```mermaid
sequenceDiagram
    participant Code as Tumhara Code
    participant WS as with statement
    participant File as File Object

    Code->>WS: with open(...) as file:
    WS->>File: file open karo
    Code->>File: read()/write() operations
    Note over Code,File: Agar exception aaye bhi to...
    WS->>File: automatically close() call hoga
    File-->>WS: File safely closed
```

---

## 1.5 Files Padhna (Reading)

4 tareeke hain file padhne ke:

### a) `read()` — poori file ek saath string mein

```python
with open("notes.txt", "r") as f:
    content = f.read()
    print(content)   # poora content ek string ki tarah
```

### b) `read(n)` — sirf `n` characters padhna

```python
with open("notes.txt", "r") as f:
    chunk = f.read(10)   # sirf pehle 10 characters
```

### c) `readline()` — ek line padhna (cursor agli line pe move ho jaata hai)

```python
with open("notes.txt", "r") as f:
    line1 = f.readline()
    line2 = f.readline()
    print(line1, line2)
```

### d) `readlines()` — saari lines ek **list** mein

```python
with open("notes.txt", "r") as f:
    lines = f.readlines()
    print(lines)   # ['line1\n', 'line2\n', 'line3\n']
```

### e) File object khud iterable hai (memory-efficient — best for bade files)

```python
with open("notes.txt", "r") as f:
    for line in f:              # ek-ek line load hoti hai, poori file ek saath nahi
        print(line.strip())     # strip() se \n hat jaata hai
```

> **Beginner tip:** Chhoti files ke liye `read()` theek hai. Bade files (jaise logs, GB size ke files) ke liye `for line in f:` use karo — ye poori file ek saath RAM mein load nahi karta.

---

## 1.6 Files Likhna (Writing)

### a) `write()` — string likhna

```python
with open("notes.txt", "w") as f:
    f.write("Hello Samarth!\n")
    f.write("Ye second line hai.\n")
```

> Dhyan do: `write()` khud se newline (`\n`) add nahi karta — tumhe manually daalna padta hai.

### b) `writelines()` — list of strings likhna

```python
lines = ["Line 1\n", "Line 2\n", "Line 3\n"]
with open("notes.txt", "w") as f:
    f.writelines(lines)
```

### c) Append mode — end mein add karna

```python
with open("notes.txt", "a") as f:
    f.write("Ye line end mein add hogi.\n")
```

```mermaid
flowchart LR
    subgraph "w mode"
    W1[Purana Content] -->|DELETE| W2[Khaali File]
    W2 --> W3[Naya Content]
    end

    subgraph "a mode"
    A1[Purana Content] --> A2["Purana Content +\nNaya Content (end mein)"]
    end
```

---

## 1.7 File Pointer — `tell()` aur `seek()`

Jab tum file padhte/likhte ho, ek invisible **cursor** hota hai jo track karta hai ki tum file mein kahaan ho. Isse **file pointer** kehte hain.

- **`tell()`** → batata hai cursor abhi kis position (byte number) pe hai
- **`seek(offset)`** → cursor ko kisi specific position pe le jaata hai

```python
with open("notes.txt", "r") as f:
    print(f.tell())        # 0 (start mein)
    data = f.read(5)
    print(f.tell())        # 5 (5 characters padhne ke baad)

    f.seek(0)               # wapas start pe le jao
    print(f.read())         # poora content phir se padh sakte ho
```

```mermaid
flowchart LR
    A["H-e-l-l-o- -W-o-r-l-d"] 
    B(("Cursor\nPosition 0"))
    B -.seek 6.-> C(("Cursor\nPosition 6\n(W se start))"))
```

**Use case:** Bade files mein baar-baar ek hi jagah se padhna, ya kisi specific position se overwrite karna (`r+` mode ke saath).

---

## 1.8 Exception Handling in File Ops

File operations mein bahut kuch galat ho sakta hai — file exist hi na kare, permission na ho, disk full ho jaaye. Isliye **try-except** zaroori hai.

```python
try:
    with open("notes.txt", "r") as f:
        content = f.read()
except FileNotFoundError:
    print("Error: File exist nahi karti!")
except PermissionError:
    print("Error: Tumhe is file ko padhne ki permission nahi hai.")
else:
    print("File successfully padh li gayi.")
finally:
    print("Ye hamesha chalega, chahe error aaye ya na aaye.")
```

**Common file-related exceptions:**

| Exception | Kab aata hai |
|---|---|
| `FileNotFoundError` | File exist nahi karti (read mode mein) |
| `PermissionError` | OS-level permission nahi hai |
| `IsADirectoryError` | Tum file ki jagah folder open karne ki koshish kar rahe ho |
| `UnicodeDecodeError` | Wrong encoding se text file padh rahe ho |

---

## 1.9 Paths ke saath kaam karna

### `os.path` module (purana tareeka)

```python
import os

print(os.path.exists("notes.txt"))     # True/False
print(os.path.join("folder", "file.txt"))  # OS ke hisaab se path banata hai
print(os.getcwd())                      # current working directory
os.remove("notes.txt")                  # file delete karna
os.rename("old.txt", "new.txt")         # rename karna
```

### `pathlib` module (modern, recommended tareeka)

```python
from pathlib import Path

p = Path("notes.txt")
print(p.exists())          # True/False
print(p.name)               # notes.txt
print(p.suffix)             # .txt
print(p.parent)             # folder jisme file hai

# Path banana — ye zyada readable hai
data_file = Path("data") / "2026" / "report.txt"
```

> **Modern Python mein `pathlib` prefer kiya jaata hai** kyunki ye zyada readable hai aur cross-platform (Windows/Linux/Mac) issues khud handle kar leta hai.

---

## 1.10 CSV Files Handle Karna

CSV (Comma Separated Values) real projects mein bahut common hai — Excel jaisa tabular data text file mein.

```python
import csv

# CSV likhna
with open("students.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Name", "Marks"])       # header
    writer.writerow(["Samarth", 92])
    writer.writerow(["Rahul", 85])

# CSV padhna
with open("students.csv", "r") as f:
    reader = csv.reader(f)
    for row in reader:
        print(row)   # ['Name', 'Marks'], ['Samarth', '92'], ...

# DictReader — column names ke saath (zyada readable)
with open("students.csv", "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(row["Name"], row["Marks"])
```

---

## 1.11 Advanced: Custom Context Managers

Tumne dekha `with open(...) as f:` kaise kaam karta hai. Tum apna khud ka context manager bhi bana sakte ho, kisi bhi resource ke liye (sirf files ke liye nahi).

```python
class MyFileHandler:
    def __init__(self, filename, mode):
        self.filename = filename
        self.mode = mode

    def __enter__(self):
        self.file = open(self.filename, self.mode)
        print("File opened!")
        return self.file

    def __exit__(self, exc_type, exc_value, traceback):
        self.file.close()
        print("File closed automatically!")

# Usage
with MyFileHandler("notes.txt", "r") as f:
    print(f.read())
```

`__enter__` chalta hai jab `with` block start hota hai, `__exit__` chalta hai jab block khatam hota hai (chahe normally ya error ki wajah se).

```mermaid
sequenceDiagram
    participant U as with block
    participant CM as Custom Context Manager

    U->>CM: __enter__() call hota hai
    CM-->>U: resource return hota hai (e.g. file)
    Note over U: block ke andar code chalta hai
    U->>CM: __exit__() call hota hai (guaranteed)
    CM-->>U: cleanup ho jaata hai
```

---

## 1.12 Advanced: Bade Files aur Buffering

Agar file bahut badi hai (jaise 5GB log file), to use poora `read()` karna RAM crash kar sakta hai. Solution: **chunks mein padho**.

```python
def process_large_file(filename, chunk_size=1024 * 1024):  # 1MB chunks
    with open(filename, "r") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break   # file khatam
            # is chunk ko process karo
            print(len(chunk))
```

**Buffering kya hai?** Python internally file writes ko turant disk pe nahi bhejta — pehle ek **buffer** (temporary memory area) mein jama karta hai, phir batch mein disk pe likhta hai. Ye performance ke liye hota hai (disk access slow hota hai, RAM access fast).

```python
with open("notes.txt", "w", buffering=1) as f:   # line-buffered
    f.write("Turant disk pe likho\n")
```

- `buffering=0` → koi buffering nahi (sirf binary mode mein)
- `buffering=1` → line buffering (text mode mein)
- `buffering > 1` → utne bytes ka buffer size

---

## 1.13 Binary Files (Images etc.)

Text mode text ke liye hai. Images, videos, ya koi bhi non-text file ke liye **binary mode (`b`)** use karo.

```python
# Image copy karna, byte-by-byte
with open("photo.jpg", "rb") as source:
    data = source.read()

with open("photo_copy.jpg", "wb") as target:
    target.write(data)
```

> Agar tum binary file ko text mode (`r`) mein khologe, to `UnicodeDecodeError` aayega — kyunki binary data valid text characters nahi hote.

---

# Part 2: Serialization & Deserialization

## 2.1 Serialization kya hai

Jab tum Python mein kaam karte ho, tumhare paas **objects** hote hain — dictionaries, lists, custom class instances. Ye sab **RAM mein Python ke internal format** mein rehte hain.

Problem: agar tumhe ye object...
- **file mein save** karna hai, ya
- **network pe kisi doosre computer ko bhejna** hai, ya
- **database mein store** karna hai...

...to tumhe usko ek aise format mein convert karna padega jo **storable ya transmittable** ho (jaise bytes ya string).

> **Serialization** = Object ko ek format (string/bytes) mein convert karna jo save ya transmit kiya ja sake.

**Real life analogy:** Socho tumhe ek **assembled almirah** (cupboard) doosre shehar bhejni hai. Poori assembled almirah truck mein nahi jaayegi. Tum usko **khol kar flat-pack** karte ho (IKEA style), box mein daal ke bhejte ho — ye hai **serialization**. Doosri taraf jaake wapas **assemble** karna — ye hai **deserialization**.

```mermaid
flowchart LR
    A["Python Object\n(dict, list, class instance)"] -->|Serialize| B["String/Bytes\n(JSON, Pickle, etc.)"]
    B -->|Save to file / Send over network| C[(Storage / Network)]
    C -->|Load back| D["String/Bytes"]
    D -->|Deserialize| E["Python Object\n(wapas original form mein)"]

    style A fill:#cce5ff
    style E fill:#cce5ff
    style B fill:#fff3cd
    style D fill:#fff3cd
```

---

## 2.2 Deserialization kya hai

**Deserialization** serialization ka **ulta process** hai — serialized data (string/bytes) ko wapas usable Python object mein convert karna.

```mermaid
flowchart TD
    Serialize["🔵 Object → String/Bytes"] -->|"Naam: json.dumps() / pickle.dumps()"| S
    S["Serialized Data"] --> Deserialize["🟢 String/Bytes → Object"]
    Deserialize -->|"Naam: json.loads() / pickle.loads()"| D["Original Object wapas mil gaya"]
```

| Direction | Naam | Kya hota hai |
|---|---|---|
| Object → String/Bytes | **Serialization** | Save/send karne layak banana |
| String/Bytes → Object | **Deserialization** | Wapas use karne layak banana |

---

## 2.3 Kyun zaroori hai

1. **Data Persistence** — Program band karne ke baad bhi data save rahe (jaise game ka save file, ya app ki settings).
2. **APIs / Web Communication** — Jab tumhara backend frontend ko data bhejta hai (ya doosre server ko), to JSON serialize karke bhejta hai, kyunki raw Python object network pe nahi bhej sakte.
3. **Caching** — Complex computation ka result serialize karke store kar do, dobara compute karne ki zaroorat nahi.
4. **Cross-language Communication** — JSON almost har language samajhti hai, isliye Python ka data JavaScript/Java ko bhi bhej sakte ho.
5. **Database Storage** — Complex objects ko database mein text/blob ki tarah store karna.

> Ye exactly wahi concept hai jo tab kaam aata hai jab koi backend API (jaise Razorpay jaisi payment gateway) request/response mein data bhejta hai — wo data hamesha **serialize** hoke jaata hai (mostly JSON format mein) aur doosri taraf **deserialize** hota hai.

---

## 2.4 JSON Serialization Deep Dive

**JSON (JavaScript Object Notation)** sabse common serialization format hai — human-readable, lightweight, aur language-independent.

Python mein `json` module built-in hai.

### Python objects ↔ JSON — Mapping Table

| Python Type | JSON Type |
|---|---|
| `dict` | `object {}` |
| `list`, `tuple` | `array []` |
| `str` | `string` |
| `int`, `float` | `number` |
| `True` / `False` | `true` / `false` |
| `None` | `null` |

### a) `dumps()` — Python object ko **JSON string** mein convert karna

```python
import json

data = {
    "name": "Samarth",
    "college": "Engineering",
    "cgpa": 8.2,
    "skills": ["Python", "SQL", "DSA"],
    "is_hackathon_participant": True
}

json_string = json.dumps(data)
print(json_string)
# Output: {"name": "Samarth", "college": "Engineering", "cgpa": 8.2, ...}
print(type(json_string))   # <class 'str'>
```

**Pretty-printing** (readable format ke liye):

```python
json_string = json.dumps(data, indent=4)
print(json_string)
```

### b) `dump()` — Directly **file mein** likhna

```python
with open("student.json", "w") as f:
    json.dump(data, f, indent=4)
```

### c) `loads()` — JSON string ko **Python object** mein wapas laana

```python
json_string = '{"name": "Samarth", "cgpa": 8.2}'
python_dict = json.loads(json_string)
print(python_dict["name"])   # Samarth
print(type(python_dict))      # <class 'dict'>
```

### d) `load()` — **file se** padh kar Python object banana

```python
with open("student.json", "r") as f:
    data = json.load(f)
    print(data["name"])
```

### Quick Naming Trick to Yaad Rakhne ke liye

```mermaid
flowchart LR
    subgraph "'s' matlab STRING"
    dumps["dumps() → string banata hai"]
    loads["loads() → string se padhta hai"]
    end
    subgraph "'s' nahi hai matlab FILE"
    dump["dump() → seedha file mein likhta hai"]
    load["load() → seedha file se padhta hai"]
    end
```

> **Trick:** Jisme `s` hai (`dumps`, `loads`) wo **string** ke saath kaam karta hai. Jisme `s` nahi hai (`dump`, `load`) wo **file object** ke saath kaam karta hai.

---

## 2.5 Custom Objects ko JSON mein Convert Karna

Problem: JSON module directly `dict`, `list`, `str`, `int` jaanta hai — par tumhari **custom class ka object** nahi jaanta.

```python
class Student:
    def __init__(self, name, cgpa):
        self.name = name
        self.cgpa = cgpa

s = Student("Samarth", 8.2)
json.dumps(s)   # ❌ TypeError: Object of type Student is not JSON serializable
```

### Solution 1: `default` parameter use karo

```python
def student_to_dict(obj):
    if isinstance(obj, Student):
        return {"name": obj.name, "cgpa": obj.cgpa}
    raise TypeError("Serializable nahi hai")

s = Student("Samarth", 8.2)
json_string = json.dumps(s, default=student_to_dict)
print(json_string)   # {"name": "Samarth", "cgpa": 8.2}
```

### Solution 2: `__dict__` use karo (simple objects ke liye shortcut)

```python
json_string = json.dumps(s.__dict__)
print(json_string)   # {"name": "Samarth", "cgpa": 8.2}
```

### Wapas Custom Object banana — `object_hook`

```python
def dict_to_student(d):
    return Student(d["name"], d["cgpa"])

json_string = '{"name": "Samarth", "cgpa": 8.2}'
student_obj = json.loads(json_string, object_hook=dict_to_student)
print(student_obj.name)   # Samarth
```

---

## 2.6 Pickle Serialization Deep Dive

JSON sirf **basic data types** handle karta hai. Par agar tumhe **kisi bhi Python object** ko serialize karna hai — including complex classes, functions, ya poore data structures — to **`pickle`** module use hota hai.

Pickle Python-specific **binary format** banata hai (human-readable nahi).

```python
import pickle

data = {"name": "Samarth", "skills": ["Python", "DSA"]}

# Serialize (Python object → binary bytes)
with open("data.pkl", "wb") as f:      # 'wb' = write binary
    pickle.dump(data, f)

# Deserialize (binary bytes → Python object)
with open("data.pkl", "rb") as f:      # 'rb' = read binary
    loaded_data = pickle.load(f)
    print(loaded_data)   # {'name': 'Samarth', 'skills': ['Python', 'DSA']}
```

**String ke roop mein bhi kar sakte ho** (`dumps`/`loads` — same naming pattern jaise JSON):

```python
byte_data = pickle.dumps(data)   # bytes object milta hai (string nahi!)
original = pickle.loads(byte_data)
```

**Pickle custom classes bhi handle kar leta hai — bina extra code ke:**

```python
class Student:
    def __init__(self, name, cgpa):
        self.name = name
        self.cgpa = cgpa

s = Student("Samarth", 8.2)

with open("student.pkl", "wb") as f:
    pickle.dump(s, f)

with open("student.pkl", "rb") as f:
    loaded_student = pickle.load(f)
    print(loaded_student.name)   # Samarth — poora object wapas mil gaya!
```

---

## 2.7 JSON vs Pickle — Comparison

```mermaid
flowchart TD
    Start([Serialization format\nchoose karna hai]) --> Q1{Data doosre language\nya system ko bhejna hai?}
    Q1 -->|Haan - API, web, cross-language| JSON["✅ JSON use karo"]
    Q1 -->|Nahi - sirf Python mein rehna hai| Q2{Sirf basic data types hain\nya complex Python objects bhi?}
    Q2 -->|Basic dict/list/str/int| JSON
    Q2 -->|Complex objects, functions, classes| Pickle["✅ Pickle use karo"]
```

| Feature | JSON | Pickle |
|---|---|---|
| **Format** | Text (human-readable) | Binary (not readable) |
| **Language Support** | Universal (JS, Java, Python, sab) | **Sirf Python** |
| **Data Types** | Basic types only (dict, list, str, int, bool, None) | Almost **koi bhi** Python object (classes, functions, sets, etc.) |
| **Security** | Safe — sirf data hai | ⚠️ **Unsafe** agar untrusted source se ho |
| **Speed** | Thoda slow (text parsing) | Fast (binary) |
| **Use Case** | APIs, config files, web communication | Internal caching, ML model saving, Python-only apps |
| **File Size** | Bada (text) | Chhota (binary) |

---

## 2.8 Security Warning ⚠️

> **KABHI BHI kisi untrusted/unknown source se aayi hui pickle file ko load mat karo.**

Kyunki Pickle deserialize karte waqt **arbitrary code execute** kar sakta hai. Agar koi malicious user ne pickle file craft ki hai, to `pickle.load()` call karte hi tumhare system pe **harmful code chal sakta hai** — bina tumhe pata chale.

```python
# ❌ DANGEROUS agar file kisi untrusted source se aayi ho
with open("suspicious_file.pkl", "rb") as f:
    data = pickle.load(f)   # ye system compromise kar sakta hai!
```

**Rule of thumb:**
- Apne khud ke generate kiye hue pickle files load karna — **safe**
- Internet se download ki hui, ya kisi anjaan user se receive ki hui pickle file load karna — **kabhi mat karo**
- Network/API communication ke liye **hamesha JSON use karo**, pickle nahi

```mermaid
flowchart LR

    A["📦 Pickle File"] --> B{"🔐 Source trusted hai?"}

    B -->|Haan - khud generate ki| C["✅ Safe to load"]
    B -->|Nahi - internet / unknown source| D["❌ NEVER load karo"]

    %% Styling
    classDef file fill:#f0e9ff,stroke:#7c3aed,color:#111827,stroke-width:2px
    classDef decision fill:#eaf2ff,stroke:#3b82f6,color:#111827,stroke-width:2px
    classDef success fill:#eaf8ef,stroke:#22c55e,color:#111827,stroke-width:2px
    classDef danger fill:#fff0f0,stroke:#ef4444,color:#111827,stroke-width:2px

    class A file
    class B decision
    class C success
    class D danger

    %% Link styling
    linkStyle default stroke:#64748b,stroke-width:2px
```

---

## 2.9 Dusre Formats — Brief Overview

| Format | Kya hai | Kab use hota hai |
|---|---|---|
| **JSON** | Text-based, key-value | APIs, configs, web (sabse common) |
| **YAML** | JSON jaisa hi, par zyada readable, indentation-based | Config files (Docker, Kubernetes, CI/CD pipelines) |
| **XML** | Tag-based structured format | Purane enterprise systems, SOAP APIs |
| **Pickle** | Python-specific binary | Python-only internal use, ML models |
| **Protocol Buffers (protobuf)** | Google ka binary format, bahut fast aur compact | High-performance microservices, gRPC |
| **MessagePack** | JSON jaisa hi par binary aur chhota | Fast APIs jaha bandwidth bachana ho |

```python
# YAML example (pip install pyyaml chahiye hoga)
import yaml
data = {"name": "Samarth", "cgpa": 8.2}
yaml_string = yaml.dump(data)
print(yaml_string)
# name: Samarth
# cgpa: 8.2
```

---

## 2.10 Real World Use Case

Ek simple example: socho ek **backend API** hai jo student ka data return karta hai.

```mermaid
sequenceDiagram
    participant Client as Client (Browser/App)
    participant Server as Backend Server
    participant DB as Database

    Client->>Server: GET /student/1 (request)
    Server->>DB: Data fetch karo (Python object mile ga)
    DB-->>Server: Student object
    Server->>Server: json.dumps() → Serialize
    Server-->>Client: JSON response bhejo
    Client->>Client: JSON.parse() → Deserialize (JS side)
    Note over Client: Ab client ke paas usable data hai
```

```python
# Backend side (Python - Flask jaisa framework)
import json

def get_student_api():
    student_object = fetch_student_from_db()   # Python object
    response_json = json.dumps({
        "name": student_object.name,
        "cgpa": student_object.cgpa
    })
    return response_json   # ye client ko bheja jaata hai
```

Yehi wo process hai jo har payment gateway, har REST API, har web app backend mein continuously ho raha hota hai — request/response cycle mein data serialize/deserialize hota rehta hai.

---

## 2.11 Best Practices Cheat Sheet

✅ **File Handling:**
- Hamesha `with` statement use karo, manual `close()` pe depend mat karo
- Bade files ke liye `for line in f:` ya chunked reading use karo
- File operations ko `try-except` mein wrap karo
- Paths ke liye `pathlib` use karo (modern aur cross-platform)

✅ **Serialization:**
- Cross-language ya network communication ke liye **JSON** use karo
- Python-only internal storage (jaise ML models, cache) ke liye **Pickle** use karo
- **Kabhi bhi untrusted pickle files load mat karo**
- Custom objects serialize karne ke liye `default=` (JSON) ya `__dict__` use karo
- Config files ke liye YAML zyada readable option hai

---

## Quick Recap — Sab Kuch Ek Nazar Mein

```mermaid
%%{init: {'theme': 'dark', 'themeVariables': {'primaryColor': '#3b4252', 'primaryTextColor': '#eceff4', 'primaryBorderColor': '#88c0d0', 'lineColor': '#88c0d0', 'secondaryColor': '#434c5e', 'tertiaryColor': '#4c566a', 'mainBkg': '#2e3440', 'textColor': '#eceff4'}}}%%
mindmap
  root((File Handling +\nSerialization))
    File Handling
      Modes: r w a r+ w+ a+
      with statement
      read/readline/readlines
      write/writelines
      seek/tell
      pathlib
      CSV
    Serialization
      JSON
        dumps/loads - string
        dump/load - file
      Pickle
        Python-only binary
        Security risk
      Other formats
        YAML
        XML
        Protobuf
```

**Bas itna samajh lo:**
- **File Handling** = data ko disk pe permanently save/read karna
- **Serialization** = object ko storable/sendable format mein convert karna
- **Deserialization** = us format ko wapas usable object mein convert karna
- **JSON** = universal, text, safe → APIs ke liye
- **Pickle** = Python-only, binary, powerful par risky → internal use ke liye

---

#### Above was a deep dive in python for complete beginners .Now it will a for those who have already studied upper one 

### Some Theory

##### Types of data used for I/O:
- Text - '12345' as a sequence of unicode chars
- Binary - 12345 as a sequence of bytes of its binary equivalent

##### Hence there are 2 file types to deal with
- Text files - All program files are text files
- Binary Files - Images,music,video,exe files

##### How File I/O is done in most programming languages

- Open a file
- Read/Write data
- Close the file


In [23]:
# Writing to a file :-

#Case1:-writing to a file if file not present like bni hi nhi hai 
f=open("sample.txt","w") #Kounsi file, kounsa mode use karoge since likhna hai to w mode 
f.write("Hello World")
f.close() #file kholi thi likhne ke liye ab close kardi ab kuch write read nhi kar skte firse kholni hogi 

#Output dikhna to nhi chahiye tha par 11 dikh rha hai and if you will check this folder to yahan sample .txt karke ek file ban gaya hoga jis folder main abhi coding ki file hai usi main hi .

#File kisi bhi location par bana skte ho 

In [24]:
#writing multiple line or multiline strings 

f=open("sample1.txt","w")
f.write("Hello World")
f.write("\n how are you ")

f.close()
#output 14 isliye aayega kyunki f.write returns number of characters written into the file 

In [25]:
#If file is already present 
f=open("sample.txt","w")
f.write("Salman bHAI")
f.close() #Now write mode se purana sab hat jayega file ka and naya content will override it 


In [26]:
#Append mode allows you to add new data to existing data in file agar file nhi hai bana dega 

with open ("sample.txt","a") as file:
    file.write("Sahi hoon yaar!") #File jaake check karoge to salman bhai override nhi hoga balki uske saaath yeh bhi jud ayega 

l=["hello\n","hi\n","how are you\n","I am Samarth\n"]

#How to writ this list with multiple thinngs 

with open ("sample.txt","a") as file:
    file.writelines(l)  # wrte multiple lines on a file and with one line matlab list main daalke kardo bhai   

In [27]:
#Reading from a file
with open ("sample.txt","r") as file:
    s=file.read()
    
    print(s)
with open ("sample.txt","r") as file:
    w=file.read(10)   
    print(w) 
    

Salman bHAISahi hoon yaar!hello
hi
how are you
I am Samarth

Salman bHA


In [28]:
#Reading line by line 
with open ("sample.txt","r") as file:
    print(file.readline())
    print(file.readline())
    # ab do line ka gap aayege kyunki readline to line change karega hai print bhi line change karega

    #Readline tab use karo jab file bahot zyada hi badi hai and ek baar main memory main load nhi karni
    # PAr readline ko utni baar call karoge jitni lines hai mujhe nhi pata kitni hai ?
    # to custom code  likho
with open ("sample.txt","r") as file:
    
    while True:
        data=file.readline()
        if data != "":
            print(data,end="") 
        else:break

Salman bHAISahi hoon yaar!hello

hi

Salman bHAISahi hoon yaar!hello
hi
how are you
I am Samarth


In [29]:
#Chunk Loading
with open("sample1.txt","w") as file:
    i=0
    while i<1000:
        file.write("hello world ")
        i+=1
with open("sample1.txt","r")as file:
    chunk=25
    data=file.read(chunk)
    while len(data)>0:
        print(data)
        data=file.read(chunk)      

hello world hello world h
ello world hello world he
llo world hello world hel
lo world hello world hell
o world hello world hello
 world hello world hello 
world hello world hello w
orld hello world hello wo
rld hello world hello wor
ld hello world hello worl
d hello world hello world
 hello world hello world 
hello world hello world h
ello world hello world he
llo world hello world hel
lo world hello world hell
o world hello world hello
 world hello world hello 
world hello world hello w
orld hello world hello wo
rld hello world hello wor
ld hello world hello worl
d hello world hello world
 hello world hello world 
hello world hello world h
ello world hello world he
llo world hello world hel
lo world hello world hell
o world hello world hello
 world hello world hello 
world hello world hello w
orld hello world hello wo
rld hello world hello wor
ld hello world hello worl
d hello world hello world
 hello world hello world 
hello world hello world h
ello world hello world he
llo world he

In [30]:
#Seek function:
#It allows you to drag the buffer like agar abhi 10th char pe hai to keech ke kahi bhi le jaa skte hoo liek 1 se 10 print kiye ab 11 se 20 honge par main chahta  hoon ki phirse 1 to 10 ho to seek use karunga 

with open ("sample.txt","r") as file:
    print(file.read(10))
    print(file.tell())
    file.seek(0)
    print(file.read(10))



Salman bHA
10
Salman bHA


In [31]:
#Tell function:
#is a mechanism to tell kitne characters process hogaye and next kounsa hoga 


with open ("sample1.txt","r") as file:
    print(file.read(10))
    print(file.tell())

hello worl
10


In [32]:
#seek during write 
with open("sample2.txt","w") as file:
    file.write("Hii my name is Samarth")
    file.seek(0)
    file.write("X")

    #Ab isse hoga kuch yu ki Hii my name is samarth sample2.txt main chala gaya then buffere seek ki wajah se 0 pe aagay ab 0 pe h hai to write h ko override karke x likh dega .
    
    #Result:Xii my name is samarth


#### Problems with working in text mode 
* can't work with binary files like images  
* not good for other datatypes like int/float/list/tuples 

In [33]:
#working wiht binary files 
#Pehle iss folder main koi image load karlo (img isi folder main rkhna jisme ye notebook hai)

#now try to read this image file
with open("photo.jpg","r") as file:
    file.read() #error aajayega(file ka extension shi se daalna bhai)

#Ye jo error hai ye binary data ke liye hai    since what we are doing is trying to read a textual file therefore it searches for unicode characters but milte hi nhi hai since file is binary  

FileNotFoundError: [Errno 2] No such file or directory: 'photo.jpg'

In [ ]:
with open("photo.jpg","rb") as file:
    #Ye method rb is for reading binary files
    with open("photo1.jpg","wb") as write_file:
        write_file.write(file.read())

 #Ab dekhna ek new file create hogi jisme original photo copy ho jayegi        

In [ ]:
#Ye to hogaya first problem ka solution ki normal modes binary files pe kaam nhi karte ab dusri problem ka solution:

#ab text file ke andar integer store karne ki mansha hai 

with open ("sample.txt","w") as file:
    file.write(5)

    #Error dedega kyunki txt file main sirf txt hi allowed hai that is unicode characters or string 


In [ ]:
#More complex datatype 
d={
    "name":"Samarth",
    "age":20,
    "gender":"Male"
}
with open ("sample.txt","w") as file:
    file.write(d)
    ##throws error ki write() ka argument must be string not dict (dictionary)!
    #agar file.write(str(d)) karoge to ho jayega bhai 
 

In [ ]:
d={
    "name":"Samarth",
    "age":20,
    "gender":"Male"
}
with open ("sample.txt","w") as file:
    file.write(str(d))

with open("sample.txt","r") as file:
    print(file.read())  
    #Dictionary hi print hogi par agar type(file.read()) kroge to type strig hi dikhayega hence hame pehle file.read() ko dictionary mainn convert karna hoga by dict(file.read())
    #par phir bhi convert nhi hoga yni agar koshish karke complex datatype daal bhi diya to original kabhi wapapis nhi milega toh koi kaam hi nhi kar paoge 

with open("sample.txt","r") as file:
    print(file.read())        

In [ ]:
#Serialization: Process of converting python data  types to universal text format known as JSON(Javascript object notation)
#Deserialization:Process of converting JSON back to python data types 

#Serializatoion using JSON module:-
#list
import json
l=[1,2,3,4]

with open("demo.json","w") as file:
    json.dump(l,file)  #dump() ek method hota hai json module main foramt is like, json.dump("kya dump karna hai" ,"kahan karna hai(filehandler object)")

#Storing dictionaries 
d={
    "name":"Samarth",
    "age":20,
    "gender":"Male"
}
with open("demo.json","w") as file:
    json.dump(d,file,indent=4)    #indent ki help se aur ache format main jaati hai dictionary file main ye json.dump () main hota hai  can be used for any type of file  
    

In [ ]:
#Desrialization using json
import json
with open("demo.json","r") as file:
    print(json.load(file))  #json.load requires file handler object to load files 
    file.seek(0) 
    d=json.load(file)
    print(type(d))
    #dekho type bhi dict aaraha hai and yaad rkhna file.read entire file read karega ir agr yni with open mainn file read karne jaoge to error aayega because head abhi file ke end mainn hai seek se usko wapis start (0) pe laana hoga  

{'name': 'Samarth', 'age': 20, 'gender': 'Male'}
<class 'dict'>


In [ ]:
#Serialize and Deserialize a tuple 
#Tuple ke  case main alag funda hai bhai 

import json
tup1=(1,2,3,4)
with open("demo.json","w") as file:
    json.dump(tup1,file)

#dump to ho jayega par dump hone ke baad wo demo.json file main as a list hi dump hoga 
# aisa kyu?? 
# Root cause: JSON format mein "tuple" naam ki koi cheez hoti hi nahi

# JSON spec (ECMA-404) sirf ye types define karta hai:

# object {}
# array []
# string
# number
# boolean
# null

# Jab tum json.dump(tup1, file) karte ho, andar JSONEncoder chalta hai, jiska ek default type-mapping table hota hai:


# python type	    |      JSON type
# ----------------------------------------
# dict	            |   object
# list aur tuple	|   array/list
# str               |   string
# int, float	    |   number
# True/False	    |   true/false
# None	            |   null




#Deserialize karte samay list hi aayega tuple main convert karna hoga
with open("demo.json","r") as file:
    print(tuple(json.load(file)))

In [ ]:
# serialize and deserialize a nested dict

d = {
    'student':'nitish',
     'marks':[23,14,34,45,56]
}

with open('demo.json','w') as f:
  json.dump(d,f)

(1, 2, 3, 4)


In [ ]:
#Serializing/Deserializing  custom objects
class Person:

  def __init__(self,fname,lname,age,gender):
    self.fname = fname
    self.lname = lname
    self.age = age
    self.gender = gender

person=Person("Samarth","Pathrotkar",20,"Male") #is object ko dump karna hai is format main ki file mainn aise dikhe # -> Nitish Singh age -> 33 gender -> male and print bhi aise hi ho

#as a string krna chahu to 
import  json
with open("demo.json","w") as file:
  json.dump(person,file)

  #error de dega kyunki readymade datatype chala lega par objects nhi hoge kyunki  json ye jaanta hai ki normal datatype kaise serialize honge par object ka nhi  pata to batana padega 

In [ ]:
import json
person=Person("Samarth","Pathrotkar",20,"Male")
def show_object(person):
    if isinstance(person,Person): # tells where first argument kisi class ka object hai ya nhi 
        return "{} {} age->{} gender->{}".format(person.fname,person.lname,person.age,person.gender)
with open("demo.json","w") as file:
  json.dump(person,file,default=show_object) #default batayega json ko ki kaise serialize karna hai     
     

In [ ]:
#Serializing object as a dict 
class Person:

  def __init__(self,fname,lname,age,gender):
    self.fname = fname
    self.lname = lname
    self.age = age
    self.gender = gender
person=Person("Samarth","Pathrotkar",20,"Male")
def show_object_As_dict(person):
   
    if isinstance(person,Person):
        return { "name":person.fname +" "+person.lname,
                "age":person.age,
                "gender":person.gender
               }
with open("demo.json","w") as file:
  json.dump(person,file,default=show_object_As_dict)   

#Deserializing:-
with open("demo.json","r") as file:
  print(json.load(file))
      

{'name': 'Samarth Pathrotkar', 'age': 20, 'gender': 'Male'}


In [ ]:
#What is mujhe object  ko as it is serialize and deserialize karna hai taaki main woh sab kar pau jo main as normal object ke saath kar pata hoon ?
#  matlab main chahta hoon is object ko uthau kisi dusri file main le jau aur waise hi use karu jaise yahan kar raha to ye json se possible nhi hai 

In [ ]:
#Ye karne ke liye binary main convert karna hoga 
#Pickling se hoga 

### Pickling
`Pickling` is the process whereby a Python object hierarchy is converted into a byte stream, and `unpickling` is the inverse operation, whereby a byte stream (from a binary file or bytes-like object) is converted back into an object hierarchy.

In [ ]:
class Person:

  def __init__(self,name,age):
    self.name = name
    self.age = age

  def display_info(self):
    print('Hi my name is',self.name,'and I am ',self.age,'years old')

p=Person("nitish", "40")

import pickle 
with open("person.pkl","wb") as file : # wb isliye kyunki hum object to byte stream yni binary main convert karke pkl file main write kar rhe hai 
  pickle.dump(p,file)
